# 05 Context Pruning Experiment (Colab)

기존 generation 결과를 덮어쓰지 않고 `experiments/context_pruning_experiment.py`를 Colab에서 실행하기 위한 노트북입니다.

흐름: GPU 확인 → 브랜치 clone/pull → 설치 → Drive mount → 입력 파일 복사 → context-only dry run → generation 실험 → 결과 Drive 저장

In [1]:
from pathlib import Path

REPO_URL = 'https://github.com/beomsookim1020/chatbot.git'
BRANCH = 'colab-generation'
PROJECT_DIR = Path('/content/chatbot')

DRIVE_INPUT_ROOT = Path('/content/drive/MyDrive/chatbot_colab_inputs')
DRIVE_OUTPUT_ROOT = Path('/content/drive/MyDrive/chatbot_colab_outputs')

PREDICTION_REL = Path('outputs/predictions/best_variant_predictions.jsonl')
EVAL_REL = Path('data/eval/representative_wrong_30_eval_batch_format.csv')
CHUNK_REL = Path('indexes/chroma_kure_v1_soyeon_690_260520_chunks_v2_690/chunks.jsonl')
SOURCE_STORE_REL = Path('data/source_store_v2_690.jsonl')

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
MAX_NEW_TOKENS = 384
RUN_LIMIT = 0  # 0이면 eval CSV 전체. 현재 기본 eval은 wrong 30개입니다.

# 비워두면 A/B/C/D 전체 variant를 실행합니다.
RUN_VARIANTS = []
# 예: RUN_VARIANTS = ['A_current_context_baseline', 'D_hierarchical_context_with_evidence_ranking']

RUN_CONTEXT_ONLY_DRY_RUN = True
CONTEXT_ONLY_DRY_RUN_LIMIT = 2
RUN_GENERATION = True

LOCAL_OUTPUT_ROOT = PROJECT_DIR / 'outputs/context_experiments'
DRIVE_EXPERIMENT_ROOT = DRIVE_OUTPUT_ROOT / 'context_experiments'

print('branch:', BRANCH)
print('model:', MODEL_NAME)
print('eval:', EVAL_REL)
print('predictions:', PREDICTION_REL)
print('chunks:', CHUNK_REL)
print('source_store:', SOURCE_STORE_REL)

branch: colab-generation
model: Qwen/Qwen2.5-3B-Instruct
eval: data/eval/representative_wrong_30_eval_batch_format.csv
predictions: outputs/predictions/best_variant_predictions.jsonl
chunks: indexes/chroma_kure_v1_soyeon_690_260520_chunks_v2_690/chunks.jsonl
source_store: data/source_store_v2_690.jsonl


## 1. GPU 확인

In [2]:
import torch

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Colab GPU가 켜져 있지 않습니다. Runtime > Change runtime type에서 GPU를 선택하세요.')

!nvidia-smi

torch: 2.11.0+cu128
cuda available: True
gpu: NVIDIA L4
Thu May 28 00:44:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   39C    P8             13W /   72W |       3MiB /  23034MiB |      0%      Default |
|                                         |                        |                

## 2. colab-generation 브랜치 clone/pull

In [3]:
import os
import subprocess

if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)

os.chdir(PROJECT_DIR)
print('cwd:', Path.cwd())
subprocess.run(['git', 'status', '--short'], check=True)

cwd: /content/chatbot


CompletedProcess(args=['git', 'status', '--short'], returncode=0)

## 3. 설치

In [4]:
%pip install -q -r requirements.txt
%pip install -q transformers accelerate sentencepiece protobuf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.5/174.5 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 110.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 120.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 143.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 139.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.1/397.1 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 127.5 MB/s eta 0:00:0000:010:01
   ━━━

## 4. Google Drive mount

In [5]:
from google.colab import drive

drive.mount('/content/drive')
print('Drive input root:', DRIVE_INPUT_ROOT)
print('Drive output root:', DRIVE_OUTPUT_ROOT)

Mounted at /content/drive
Drive input root: /content/drive/MyDrive/chatbot_colab_inputs
Drive output root: /content/drive/MyDrive/chatbot_colab_outputs


## 5. 입력 파일 경로 검증 및 Colab 로컬 복사

In [6]:
import shutil

required_inputs = [
    ('eval', DRIVE_INPUT_ROOT / EVAL_REL),
    ('predictions', DRIVE_INPUT_ROOT / PREDICTION_REL),
    ('chunks', DRIVE_INPUT_ROOT / CHUNK_REL),
    ('source_store', DRIVE_INPUT_ROOT / SOURCE_STORE_REL),
]
missing = [(name, path) for name, path in required_inputs if not path.exists()]
if missing:
    detail = '\n'.join(f'- {name}: {path}' for name, path in missing)
    raise FileNotFoundError('Drive 입력 파일이 없습니다. 경로를 먼저 확인하세요.\n' + detail)

copy_pairs = [
    (DRIVE_INPUT_ROOT / EVAL_REL, PROJECT_DIR / EVAL_REL),
    (DRIVE_INPUT_ROOT / PREDICTION_REL, PROJECT_DIR / PREDICTION_REL),
    (DRIVE_INPUT_ROOT / CHUNK_REL, PROJECT_DIR / CHUNK_REL),
    (DRIVE_INPUT_ROOT / SOURCE_STORE_REL, PROJECT_DIR / SOURCE_STORE_REL),
]
for src, dst in copy_pairs:
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print(f'copied: {src} -> {dst} ({dst.stat().st_size:,} bytes)')

script_path = PROJECT_DIR / 'experiments/context_pruning_experiment.py'
if not script_path.exists():
    raise FileNotFoundError(f'실험 스크립트가 없습니다. colab-generation 브랜치에 새 파일이 반영됐는지 확인하세요: {script_path}')
print('script:', script_path)

copied: /content/drive/MyDrive/chatbot_colab_inputs/data/eval/representative_wrong_30_eval_batch_format.csv -> /content/chatbot/data/eval/representative_wrong_30_eval_batch_format.csv (20,087 bytes)
copied: /content/drive/MyDrive/chatbot_colab_inputs/outputs/predictions/best_variant_predictions.jsonl -> /content/chatbot/outputs/predictions/best_variant_predictions.jsonl (8,222,469 bytes)
copied: /content/drive/MyDrive/chatbot_colab_inputs/indexes/chroma_kure_v1_soyeon_690_260520_chunks_v2_690/chunks.jsonl -> /content/chatbot/indexes/chroma_kure_v1_soyeon_690_260520_chunks_v2_690/chunks.jsonl (368,289,817 bytes)
copied: /content/drive/MyDrive/chatbot_colab_inputs/data/source_store_v2_690.jsonl -> /content/chatbot/data/source_store_v2_690.jsonl (553,876,319 bytes)
script: /content/chatbot/experiments/context_pruning_experiment.py


## 6. 실행 함수

In [7]:
import sys
from datetime import datetime

CREATED_OUTPUT_DIRS = []

def list_context_outputs():
    LOCAL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    return {path.resolve() for path in LOCAL_OUTPUT_ROOT.glob('context_pruning_*') if path.is_dir()}

def run_context_experiment(*, context_only: bool, limit: int, run_name: str):
    before = list_context_outputs()
    cmd = [
        sys.executable,
        str(PROJECT_DIR / 'experiments/context_pruning_experiment.py'),
        '--predictions', str(PREDICTION_REL),
        '--eval-csv', str(EVAL_REL),
        '--chunks', str(CHUNK_REL),
        '--source-store', str(SOURCE_STORE_REL),
        '--output-root', 'outputs/context_experiments',
        '--run-name', run_name,
        '--model-name', MODEL_NAME,
        '--max-new-tokens', str(MAX_NEW_TOKENS),
        '--limit', str(limit),
    ]
    if context_only:
        cmd.append('--context-only')
    for variant in RUN_VARIANTS:
        cmd.extend(['--variant', variant])
    print('Running command:')
    print(' '.join(cmd))
    subprocess.run(cmd, cwd=PROJECT_DIR, check=True)
    after = list_context_outputs()
    created = sorted(after - before, key=lambda path: path.stat().st_mtime)
    if not created:
        raise RuntimeError('새 output directory를 찾지 못했습니다.')
    CREATED_OUTPUT_DIRS.extend(created)
    print('created output:', created[-1])
    return created[-1]

## 7. Context-only dry run

모델을 로드하지 않고 context 구성만 먼저 확인합니다.

In [8]:
if RUN_CONTEXT_ONLY_DRY_RUN:
    dry_output_dir = run_context_experiment(
        context_only=True,
        limit=CONTEXT_ONLY_DRY_RUN_LIMIT,
        run_name='context_pruning_dry_run',
    )
else:
    dry_output_dir = None
    print('Context-only dry run skipped.')

Running command:
/usr/bin/python3 /content/chatbot/experiments/context_pruning_experiment.py --predictions outputs/predictions/best_variant_predictions.jsonl --eval-csv data/eval/representative_wrong_30_eval_batch_format.csv --chunks indexes/chroma_kure_v1_soyeon_690_260520_chunks_v2_690/chunks.jsonl --source-store data/source_store_v2_690.jsonl --output-root outputs/context_experiments --run-name context_pruning_dry_run --model-name Qwen/Qwen2.5-3B-Instruct --max-new-tokens 384 --limit 2 --context-only
created output: /content/chatbot/outputs/context_experiments/context_pruning_dry_run_20260528_004603


## 8. Generation 실험 실행

In [9]:
if RUN_GENERATION:
    generation_output_dir = run_context_experiment(
        context_only=False,
        limit=RUN_LIMIT,
        run_name='context_pruning',
    )
else:
    generation_output_dir = None
    print('Generation skipped.')

Running command:
/usr/bin/python3 /content/chatbot/experiments/context_pruning_experiment.py --predictions outputs/predictions/best_variant_predictions.jsonl --eval-csv data/eval/representative_wrong_30_eval_batch_format.csv --chunks indexes/chroma_kure_v1_soyeon_690_260520_chunks_v2_690/chunks.jsonl --source-store data/source_store_v2_690.jsonl --output-root outputs/context_experiments --run-name context_pruning --model-name Qwen/Qwen2.5-3B-Instruct --max-new-tokens 384 --limit 0
created output: /content/chatbot/outputs/context_experiments/context_pruning_20260528_004622


## 9. 결과 확인

In [10]:
import csv
import html as html_lib
from IPython.display import HTML, display

latest_output_dir = generation_output_dir or dry_output_dir
if latest_output_dir is None:
    raise RuntimeError('??? output directory? ????.')

metrics_path = latest_output_dir / 'context_pruning_metrics.csv'
review_path = latest_output_dir / 'context_pruning_review.csv'
summary_path = latest_output_dir / 'context_pruning_summary.md'
results_path = latest_output_dir / 'context_pruning_results.jsonl'

print('results:', results_path)
print('review:', review_path)
print('metrics:', metrics_path)
print('summary:', summary_path)

def read_csv_preview(path, limit=None):
    with path.open('r', encoding='utf-8-sig', newline='') as f:
        reader = csv.DictReader(f)
        rows = []
        for idx, row in enumerate(reader):
            if limit is not None and idx >= limit:
                break
            rows.append(row)
    return rows

def display_rows(rows, title, max_cols=12):
    print(f'{title}: {len(rows)} row(s) shown')
    if not rows:
        return
    columns = list(rows[0].keys())[:max_cols]
    header = ''.join(f'<th>{html_lib.escape(col)}</th>' for col in columns)
    body = []
    for row in rows:
        cells = ''.join(
            '<td style="max-width:260px;white-space:nowrap;overflow:hidden;text-overflow:ellipsis">'
            + html_lib.escape(str(row.get(col, ''))) + '</td>'
            for col in columns
        )
        body.append(f'<tr>{cells}</tr>')
    display(HTML(
        '<div style="overflow:auto;max-height:420px">'
        f'<table border="1" style="border-collapse:collapse;font-size:12px">'
        f'<thead><tr>{header}</tr></thead><tbody>{"".join(body)}</tbody></table>'
        '</div>'
    ))

metrics_rows = read_csv_preview(metrics_path)
review_rows = read_csv_preview(review_path, limit=10)
display_rows(metrics_rows, 'metrics')
display_rows(review_rows, 'review preview')


results: /content/chatbot/outputs/context_experiments/context_pruning_20260528_004622/context_pruning_results.jsonl
review: /content/chatbot/outputs/context_experiments/context_pruning_20260528_004622/context_pruning_review.csv
metrics: /content/chatbot/outputs/context_experiments/context_pruning_20260528_004622/context_pruning_metrics.csv
summary: /content/chatbot/outputs/context_experiments/context_pruning_20260528_004622/context_pruning_summary.md
metrics: 4 row(s) shown


answer_available_rate,answerable_rate,avg_context_chars,avg_direct_evidence_count,avg_noisy_evidence_count,avg_reference_evidence_count,avg_source_store_items,avg_supporting_evidence_count,citation_valid_rate,context_char_reduction_vs_baseline_pct,failure_tag_counts,gold_signal_present_rate
1.0,0.6,8519.633333333333,7.166666666666667,2.6666666666666665,0.0,0.0,0.0,1.0,0.0,"{""target_doc_coverage_missing"": 5, ""multi_intent_incomplete"": 4, ""llm_hallucination_risk"": 3, ""source_numeric_missing"": 12, ""llm_invalid_json"": 7, ""gt_expected_answer_but_model_not_found"": 1, ""citation_wrong_target"": 5, ""gt_numeric_mismatch"": 3, ""unit_conversion_corrected"": 3, ""gt_semantic_overlap_low"": 3, ""incomplete_multi_doc"": 1, ""wrong_target_field_selection"": 2, ""question_aspect_incomplete"": 2}",0.16666666666666666
1.0,0.5666666666666667,4852.333333333333,3.2666666666666666,1.1666666666666667,0.5666666666666667,0.0,1.1,0.9666666666666667,43.0452797264358,"{""citation_wrong_target"": 9, ""target_doc_coverage_missing"": 5, ""llm_invalid_json"": 4, ""multi_intent_incomplete"": 4, ""wrong_target_field_selection"": 4, ""llm_hallucination_risk"": 3, ""source_numeric_missing"": 12, ""incomplete_multi_doc"": 3, ""unit_conversion_corrected"": 4, ""gt_numeric_mismatch"": 3, ""insufficient_evidence"": 1, ""gt_semantic_overlap_low"": 3, ""question_aspect_incomplete"": 1}",0.16666666666666666
1.0,0.5333333333333333,5594.1,3.2666666666666666,1.4666666666666666,0.7666666666666667,1.2,1.1,0.9333333333333333,34.3387234974901,"{""citation_wrong_target"": 8, ""target_doc_coverage_missing"": 5, ""llm_invalid_json"": 2, ""insufficient_evidence"": 2, ""multi_intent_incomplete"": 4, ""wrong_target_field_selection"": 3, ""llm_hallucination_risk"": 3, ""source_numeric_missing"": 12, ""incomplete_multi_doc"": 3, ""gt_numeric_mismatch"": 3, ""unit_conversion_corrected"": 3, ""gt_semantic_overlap_low"": 2, ""gt_expected_answer_but_model_not_found"": 1, ""question_aspect_incomplete"": 1}",0.16666666666666666
1.0,0.5333333333333333,6586.333333333333,3.9,2.2,1.3666666666666667,1.4333333333333333,1.2333333333333334,0.9666666666666667,22.692291139290035,"{""target_doc_coverage_missing"": 5, ""llm_invalid_json"": 5, ""multi_intent_incomplete"": 4, ""citation_wrong_target"": 8, ""wrong_target_field_selection"": 4, ""llm_hallucination_risk"": 3, ""source_numeric_missing"": 12, ""incomplete_multi_doc"": 3, ""gt_numeric_mismatch"": 3, ""unit_conversion_corrected"": 3, ""insufficient_evidence"": 1, ""gt_semantic_overlap_low"": 3, ""question_aspect_incomplete"": 1}",0.16666666666666666


review preview: 10 row(s) shown


context_char_count,failure_tags,failure_type,generated_answer,gold_answer,gold_signal_present,manual_correct,question,question_id,question_type,retrieved_docs_top5,review_note
7552,[],,콜센터 구축 외에 함께 도입되는 통신 환경의 조건은 문서 내에서 명시되어 있지 않다.,응급실 섭외 지연 문제를 해결하기 위해 전국 병원 간 전원 및 재난 조정 업무를 위한 콜센터 구축과 함께 통신장비 도입을 조건으로 포함하고 있습니다.,False,,국립중앙의료원의 차세대 응급의료 상황관리시스템 구축 위탁용역에서 응급실 섭외 지연 문제를 해결하기 위해 제시한 콜센터 구축 외에 함께 도입되는 통신 환경의 조건은 무엇인가요?,Q006,business_type,"[""국립중앙의료원_(긴급)「2024년도 차세대 응급의료 상황관리시스템 구축.hwp"", ""국립중앙의료원_대한민국 해외긴급구호대(KDRT) 이동식병원 통합의료정.hwp"", ""국립중앙의료원_(긴급)「2024년도 스마트의료지도시스템 고도화」.hwp"", ""국립중앙의료원_병원정보시스템 노후 전산장비 교체(증설) 및 운영환경 .hwp"", ""국립중앙의료원_기부금 협력연구 연구행정통합시스템 구축 용역(재공고.hwp""]",
9643,[],,총괄 책임자의 정보보안기사 및 정보처리기사 자격증 원본 제출은 필수사항이 아닙니다.,해당 문서에는 총괄 담당자의 정보보안기사 또는 정보처리기사 자격증 필수 제출이나 요구 여부에 대한 명시적인 지침 또는 제한 정보가 포함되어 있지 않습니다.,True,,고려대학교의 '차세대 포털·학사 정보시스템 구축사업'을 수주하기 위해서는 총괄 책임자가 반드시 정보보안기사 및 정보처리기사 자격증 원본을 필수로 함께 제출해야 합니까?,Q017,summary,"[""고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf"", ""고려대학교_차세대 포털·학사 정보시스템 구축 사업 재공고.pdf"", ""고려대학교_고려대학교 차세대 LMS(학습관리시스템) 구축 사업.pdf"", ""고려대학교_[재공지] 고려대학교 공간관리 통합시스템 구축 사업.hwp"", ""고려대학교_고려대학교 공간관리 통합시스템 구축 사업.hwp""]",
4238,"[""target_doc_coverage_missing""]",,주된 기대효과는 기후 변화 대응 스마트 관개 시스템 구축을 통해 우즈bekistan과 kyrgyzstan 간의 상호 협력과 기후 변화 대응 능력을 향상시키는 것이다.,"해당 스마트 관개시스템 구축사업의 주요 성과 목표 및 기대효과로는 지역적 기후 회복력 강화, 홍수 및 가뭄 관리에 혁신적인 스마트 기술 도입, 그리고 소득 창출 및 녹색 일자리 창출 개선 등이 있습니다.",False,,아시아물위원훼의 우즈벡-키르기즈쓰탄 슴아트 관계 시스템의 주된 기대효괍는 무엇인지 알려주새요.,Q020,general,"[""사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화대응 스.hwp""]",
11994,"[""multi_intent_incomplete"", ""target_doc_coverage_missing""]",,"국립중앙의료원의 위탁용역은 고도화, 구축, 운영, 보안 등의 사업유형을 가지고 있으며, 그랜드코리아레저(주)의 그룹웨어 구축 용역은 그룹웨어 시스템 구축에 대한 사업이다. 두 사업 모두 정보시스템을 구축하고 운영하는데 초점을 맞추고 있지만, 구체적인 업무 목적과 범위는 다르다.","국립중앙의료원 사업은 병원 간 응급환자 전원 상황 시 섭외 대기 지연 문제를 극복하기 위한 '전국 콜센터 연계 상황관리망' 구축이 목적이며, 그랜드코리아레저(주) 사업은 노후화된 업무 환경의 연속성을 확보하고자 정부 지침에 부합하는 '웹 기반 통합 기록물 및 메신저 체제 그룹웨어' 시스템 구축을 목적으로 하고 있습니다.",False,,국립중앙의료원의 위탁용역과 그랜드코리아레저(주)의 그룹웨어 구축 용역에서 각각 핵심으로 다루는 정보시스템의 업무 목적과 범위를 비교해 주십시오.,Q031,summary,"[""그랜드코리아레저(주)_2024년도 GKL 그룹웨어 시스템 구축 용역.hwp"", ""한국농수산식품유통공사_통합정보시스템 위탁운영 용역.hwp"", ""국립중앙의료원_(긴급)「2024년도 차세대 응급의료 상황관리시스템 구축.hwp"", ""국립중앙의료원_기부금 협력연구 연구행정통합시스템 구축 용역(재공고.hwp"", ""국립중앙의료원_(긴급)「2024년도 스마트의료지도시스템 고도화」.hwp""]",
11924,[],,ISMP는 정보시스템 고도화 계획을 수립하는 것을 지칭합니다.,공고된 용역명 내의 ISMP는 곧장 '업무정보화종합계획'을 수립하는 것을 대등하게 지칭하고 있습니다.,True,,(주)에스알에서 발주한 '예약발매시스템 개량 ISMP 용역' 사업의 공식 과업명에서 ISMP는 어떤 계획을 수립하는 것을 지칭합니까?,Q123,summary,"[""한국철도공사 (용역)_예약발매시스템 개량 ISMP 용역.hwp"", ""한국철도공사 (용역)_중장기 정보화전략계획(ISP) 수정·보완 용역.hwp"", ""한국철도공사 (용역)_[긴급][협상형]운행정보기록 자동분석시스템 개량.hwp"", ""한국철도공사 (용역)_감사정보시스템 고도화 ISP 용역.hwp"", ""한국철도공사 (용역)_모바일오피스 시스템 고도화 용역(총체 및 1차).hwp""]",
10528,"[""llm_hallucination_risk"", ""source_numeric_missing""]",,무상 하자담보 책임 기간은 계약 종료 후 2년입니다.,"본 기관에서 제시한 산학협력단 정보시스템 운영 입찰 제안 가이드 및 내용 전반을 통틀어, 사업 종료 시점 이후 수급자가 강제로 부담해야 하는 무상 하자담보 보수 기간(예: 1년 또는 2년 등)에 대한 정량적 조항은 존재하지 않습니다.",False,,경희대학교 산학협력단의 정보시스템 유지보수 용역을 수주한 업체가 무상 하자담보 책임을 져야 하는 법적 기간은 계약 종료 후 몇 년입니까?,Q137,summary,"[""경희대학교_[입찰공고] 2024~2025학년도 경희대학교 정보시스템(인포21 등) .hwp"", ""경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp"", ""경희대학교_[재공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp"", ""경희대학교_[재공고] 2024~2025학년도 경희대학교 정보시스템(인포21 등) 운.hwp"", ""경희대학교_[입찰공고] 경희대학교 공간관리시스템 구축 용역업체 선정.hwp""]",
5015,"[""source_numeric_missing""]",,한국가스공사 차세대 통합정보시스템(ERP) 구축의 사업예산 근거를 context에서 확인할 수 없어 계산할 수 없습니다.,"해당 사업의 예산 규모는 14,107,009,000원입니다.",False,,한국가스공사의 '차세대 통합정보시스템(ERP) 구축' 사업 예산 규모는 얼마입니까?,Q001,budget,"[""한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp""]",
11306,"[""llm_invalid_json"", ""source_numeric_missing"", ""target_doc_coverage_missing""]",,고려대학교 차세대 포털·학사 정보시스템의 사업예산 근거를 context에서 확인할 수 없어 계산할 수 없습니다.,"고려대학교는 '정보서비스 접근성 및 품질 강화와 분산된 시스템 통합'(예산 11,270,000,000원), 그랜드코리아레저는 '노후화된 기록물관리 및 사내SNS 통합을 위한 웹기반 그룹웨어 도입'(예산 1,515,000,000원), 인천광역시는 '수작업 처리로 인한 위원회 업무 복잡성 해결 및 자료 관리 보안 문제 개선'(예산 150,000,000원)을 최우선 개선 영역으로 삼고 있습니다.",False,,"고려대학교의 '차세대 포털·학사 정보시스템', 그랜드코리아레저(주)의 '그룹웨어 시스템', 인천광역시의 '통합관리시스템 구축' 세 기관의 정보화 구축 사업에서 각 플랫폼이 가장 우선시하는

## 10. Drive로 결과 저장

In [11]:
DRIVE_EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)

copied_dirs = []
for local_dir in CREATED_OUTPUT_DIRS:
    dst = DRIVE_EXPERIMENT_ROOT / local_dir.name
    if dst.exists():
        raise FileExistsError(f'Drive 결과 폴더가 이미 있습니다. 덮어쓰지 않습니다: {dst}')
    shutil.copytree(local_dir, dst)
    copied_dirs.append(dst)
    print('copied output to Drive:', dst)

if not copied_dirs:
    print('복사할 새 결과 폴더가 없습니다.')
else:
    print('Drive output dirs:')
    for path in copied_dirs:
        print('-', path)

copied output to Drive: /content/drive/MyDrive/chatbot_colab_outputs/context_experiments/context_pruning_dry_run_20260528_004603
copied output to Drive: /content/drive/MyDrive/chatbot_colab_outputs/context_experiments/context_pruning_20260528_004622
Drive output dirs:
- /content/drive/MyDrive/chatbot_colab_outputs/context_experiments/context_pruning_dry_run_20260528_004603
- /content/drive/MyDrive/chatbot_colab_outputs/context_experiments/context_pruning_20260528_004622


## 사람이 볼 파일

- `context_pruning_review.csv`: 사람이 `manual_correct`, `failure_type`, `review_note`를 채우는 파일
- `context_pruning_metrics.csv`: variant별 context 길이, gold signal proxy, source_store 사용률 비교
- `context_pruning_summary.md`: 기존 구현 여부와 실패 사례 요약
- `context_pruning_results.jsonl`: 전체 generation 상세 결과와 `used_context` 포함